In [1]:
import requests

url = "https://api.clickup.com/api/v2/list/901202319556/task?subtasks=true"
headers = {
    "Authorization": "pk_93882701_5X39IECQRBNWFTVZBYSJH5CXKU79HS55"
}

r = requests.get(url, headers=headers)
print(r.status_code)
print(r.json())

200
{'tasks': [{'id': '869btbuwk', 'custom_id': None, 'custom_item_id': 0, 'name': 'SnowPro Advanced: Data Engineer', 'text_content': '', 'description': '', 'status': {'status': 'to do', 'id': 'p90100293020_Qr12E7Qm', 'color': '#87909e', 'type': 'open', 'orderindex': 0}, 'orderindex': '237630806.00000000000000000000000000000000', 'date_created': '1768454966454', 'date_updated': '1770259419370', 'date_closed': None, 'date_done': None, 'archived': False, 'creator': {'id': 56480542, 'username': 'Azabenathi Pupuma', 'color': '#b388ff', 'email': 'azabenathi.pupuma@slipstreamdata.co.za', 'profilePicture': None}, 'assignees': [{'id': 56480542, 'username': 'Azabenathi Pupuma', 'color': '#b388ff', 'initials': 'AP', 'email': 'azabenathi.pupuma@slipstreamdata.co.za', 'profilePicture': None}], 'group_assignees': [], 'watchers': [{'id': 50688464, 'username': 'Flora', 'color': '#795548', 'initials': 'F', 'email': 'flora.kundaeli@slipstreamdata.co.za', 'profilePicture': None}, {'id': 56480542, 'usern

In [2]:
import pandas as pd
try:
    api_json_data = r.json()

    if isinstance(api_json_data, list):
        df_api = pd.DataFrame(api_json_data)
        print("DataFrame from API list response:")
        print(df_api)
    elif isinstance(api_json_data, dict):
        # Check if the dictionary contains a 'tasks' key and if its value is a list
        if 'tasks' in api_json_data and isinstance(api_json_data['tasks'], list):
            df_api = pd.DataFrame(api_json_data['tasks'])
            print("DataFrame from API 'tasks' list within dictionary response:")
            print(df_api)
        else:
            # If it's a dictionary but not with a 'tasks' key, or 'tasks' isn't a list
            # Assume it's a single record or an error message and try to convert as is
            df_api = pd.DataFrame([api_json_data])
            print("DataFrame from API dictionary response (single record or error):")
            print(df_api)
    else:
        print("API response is not a recognized JSON format (list or dict) for DataFrame conversion.")
except NameError:
    print("Error: 'r' (the response object from the API call) is not defined in this scope. Please ensure the API call in the previous cell has been executed successfully and 'r' is available, or re-run it.")
except Exception as e:
    print(f"An error occurred during API JSON processing: {e}")

DataFrame from API 'tasks' list within dictionary response:
           id custom_id  custom_item_id  \
0   869btbuwk      None               0   
1   869ahmf7u      None               0   
2   869ahmcqj      None               0   
3   869agp147      None               0   
4   869agp0uz      None               0   
5   869af83h1      None               0   
6   869af83c0      None               0   
7   869af829t      None               0   
8   869af7qma      None               0   
9   869af7nme      None               0   
10  869ab9g7u      None               0   
11  869ab1fx3      None               0   
12  869aakx8b      None               0   
13  869aak1e3      None               0   
14  869aafmh1      None               0   
15  8698rmb2r      None               0   
16  86979jffy      None               0   
17  8695242w4      None               0   
18  8694x2b1k      None               0   
19  8694ubyqx      None               0   
20  8694hx7f1      None              

In [3]:
df_api['custom_fields']

,custom_fields
0,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
1,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
2,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
3,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
4,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
5,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
6,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
7,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
8,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...
9,[{'id': '0bf35fbd-1f77-4add-b258-62eeb2b1fe8e'...


In [16]:
import pandas as pd
import ast

# 1. Make a copy
df = df_api.copy()

# 2. If custom_fields is stored as text instead of real Python objects, convert it
def parse_if_string(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return x
    return x

df["custom_fields"] = df["custom_fields"].apply(parse_if_string)

# 3. Recursive function to flatten dictionaries
def flatten_dict(d, parent_key="", sep="_"):
    items = {}
    if not isinstance(d, dict):
        return items

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v

    return items

# 4. Flatten each row's custom_fields list into columns
def flatten_custom_fields(custom_fields):
    row_out = {}

    if not isinstance(custom_fields, list):
        return row_out

    for i, field in enumerate(custom_fields):
        if isinstance(field, dict):
            flat_field = flatten_dict(field, parent_key=f"custom_fields_{i}")
            row_out.update(flat_field)

    return row_out

# 5. Apply row by row
custom_fields_expanded = df["custom_fields"].apply(flatten_custom_fields)

# 6. Convert to dataframe
custom_fields_df = pd.DataFrame(custom_fields_expanded.tolist())

# 7. Join back to original dataframe
df_final = pd.concat([df.drop(columns=["custom_fields"]), custom_fields_df], axis=1)

# View result
print(df_final.head())
print(df_final.columns.tolist())

          id custom_id  custom_item_id  \
0  869btbuwk      None               0   
1  869ahmf7u      None               0   
2  869ahmcqj      None               0   
3  869agp147      None               0   
4  869agp0uz      None               0   

                                         name text_content description  \
0             SnowPro Advanced: Data Engineer                            
1  Salesforce Certified Agentforce Specialist                            
2   Salesforce Certified Mulesoft Developer 1                            
3                Snowflake Sales Professional                            
4      Snowflake Technical Sales Professional                            

                                              status  \
0  {'status': 'to do', 'id': 'p90100293020_Qr12E7...   
1  {'status': 'in progress', 'id': 'p90100293020_...   
2  {'status': 'in progress', 'id': 'p90100293020_...   
3  {'status': 'in progress', 'id': 'p90100293020_...   
4  {'status': 'in prog

In [17]:
display(df_final.filter(like='custom_fields_'))

,custom_fields_0_id,custom_fields_0_name,custom_fields_0_type,custom_fields_0_type_config_end,custom_fields_0_type_config_start,custom_fields_0_date_created,custom_fields_0_hide_from_guests,custom_fields_0_value_current,custom_fields_0_value_percent_completed,custom_fields_0_required,custom_fields_1_id,custom_fields_1_name,custom_fields_1_type,custom_fields_1_date_created,custom_fields_1_hide_from_guests,custom_fields_1_required,custom_fields_0_value_richtext
0,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
1,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
2,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
3,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
4,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
5,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
6,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
7,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
8,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN
9,0bf35fbd-1f77-4add-b258-62eeb2b1fe8e,% Complete,manual_progress,100,0,1693407976021,False,0,0.00,False,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False,NaN


In [7]:
df_final["custom_fields_1_date_created",]

,custom_fields_1_date_created
0,1719928804380
1,1719928804380
2,1719928804380
3,1719928804380
4,1719928804380
5,1719928804380
6,1719928804380
7,1719928804380
8,1719928804380
9,1719928804380


In [8]:
display(df_final.filter(like='custom_fields_1_'))

,custom_fields_1_id,custom_fields_1_name,custom_fields_1_type,custom_fields_1_date_created,custom_fields_1_hide_from_guests,custom_fields_1_required
0,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
1,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
2,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
3,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
4,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
5,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
6,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
7,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
8,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False
9,5391a196-5257-4828-8015-8b4d31bc6e9d,Expiration Date,date,1719928804380,False,False


In [15]:
print(df_final['custom_fields_1_date_created'].unique())

<DatetimeArray>
['2024-07-02 14:00:04.380000']
Length: 1, dtype: datetime64[ns]


In [12]:
df_final['custom_fields_1_date_created'] = pd.to_datetime(df_final['custom_fields_1_date_created'], unit='ms')
display(df_final[['name', 'custom_fields_1_name', 'custom_fields_1_date_created']].head())
display(df_final[['custom_fields_1_date_created']].info())

,name,custom_fields_1_name,custom_fields_1_date_created
0,SnowPro Advanced: Data Engineer,Expiration Date,2024-07-02 14:00:04.380
1,Salesforce Certified Agentforce Specialist,Expiration Date,2024-07-02 14:00:04.380
2,Salesforce Certified Mulesoft Developer 1,Expiration Date,2024-07-02 14:00:04.380
3,Snowflake Sales Professional,Expiration Date,2024-07-02 14:00:04.380
4,Snowflake Technical Sales Professional,Expiration Date,2024-07-02 14:00:04.380


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 1 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   custom_fields_1_date_created  32 non-null     datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 388.0 bytes


None